inconsistent documentation for now


the project will be as following <br>
1 DATA CLEANING <br>
1.1 Loading the dataset and importing the libraries<br>
1.2 missing values <br>
1.3 imputation using data from the names (not best practice)<br>
1.4 dropping missing values to avoid imputation involving categorical data<br>
1.5 binary econcoding categorical data<br>
1.6 gini index<br>
1.7 information gain<br>
2 DATA VIZUALIZATION
2.1 <br>

<h4 align = "center"> Loading the dataset and importing the libraries</h4>


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import category_encoders as ce
from copy import deepcopy

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_regression
from sklearn.pipeline import Pipeline


file_path = r"D:\ML\videogames_predictions\video games sales.csv"

df = pd.read_csv(file_path, encoding='ISO-8859-1')


In [ ]:
df.head()

In [ ]:
df.info()

<h4 align = "center"> 1.1 Counting missing values and removing the mistake entry</h4>

since the dataset contains 10K+ entries, the first step will be understanding how much of it is actually useable.<br>


In [ ]:
print(df.isnull().sum())


In [ ]:
df.loc[[df["Year"].idxmax()]] #if you add code below make sure to print this.

In [ ]:
df = df.drop(df["Year"].idxmax())

<h4 align = "center"> 1.2 simple imputation using the names to complete the years </h4>

the code below creates a temporary df that will be used to extract the years from the game names that end with a year assuming the game was published $\pm$ 1 year away from the one in the name<br>
the assumption was made because the games that end with a year are mostly sport games, note that games like the more recent "Cyberpunk 2077" could introduce an outlier into the data, resulting is worse models.

In [ ]:
null_rows = df[df.isnull().any(axis=1)]

#slices each name and checks is the last 4 characters are digits
null_rows = null_rows[null_rows['Name'].str[-4:].str.isdigit() == True]

with pd.option_context('display.max_rows', None, 'display.max_columns', None): print(null_rows) #allows you to see the whole df without the annoying .tostring formatting

name_dict will store the games names as keys and the approximated years extracted as values, the dict will be used as a temporary way to store the data before being added to the main dataframe.

In [ ]:
name_dict = {
    row: row[-4:]
    for row in null_rows['Name']
    if str(row)[-4:].isdigit() and len(str(row)) >= 4
}
print(name_dict)

filling the empty data with approximated release years

In [ ]:
df['Year'] = df['Year'].fillna(df['Name'].map(name_dict))
print(df.isnull().sum())
print( f"total number of dirty rows is {df.isnull().any(axis=1).sum()}")

originally there were 271 missing values for the "Year" column, now there are 256, 15 entries were added but the publisher column stayed the same.

In [ ]:
count = df.query("Publisher == 'Unknown'").shape[0]
print(count)

<h4 align = "center"> 1.3 Treating the rest of the missing values</h4>

the missing name of publishers will be filled by the imputer later on in the pipeline, for now the Unknown category will be extended and used as a way to store games from publishers with 3 or less games, so that we can make proper

In [ ]:
print((df['Platform'].value_counts() <= 3).sum())
print((df['Publisher'].value_counts() <= 3).sum())

platform_counts = df['Platform'].value_counts()
publisher_counts = df['Publisher'].value_counts()

df['Platform'] = df['Platform'].map(lambda x: 'Other' if (pd.isna(x) or platform_counts[x] <= 3) else x)
df['Publisher'] = df['Publisher'].map(lambda x: 'Unknown' if( pd.isna(x) or publisher_counts[x] <= 3) else x)

#quantities
print((df['Platform'] == "Other").sum())
print((df['Publisher'] == "Unknown").sum())

#reassuring the values got transformed
print((df['Platform'].value_counts() <= 3).sum())
print((df['Publisher'].value_counts() <= 3 ).sum())

comparing the combined small game publishers with a big game publisher we find out that the games are mostly similar

In [ ]:
unknown_publisher = np.array([df.loc[df['Publisher'] == 'Unknown', 'Global_Sales']])
std = unknown_publisher.std()
mean = unknown_publisher.mean()
print(std,mean)

activision = np.array([df.loc[df['Publisher'] == 'Activision', 'Global_Sales']])
std = activision.std()
mean = activision.mean()
print(std,mean)

much better situation for the platform but as before, smaller deviation than industry standards

In [ ]:
unknown_platform = np.array([df.loc[df['Platform'] == 'Other', 'Global_Sales']])
std = unknown_platform.std()
mean = unknown_platform.mean()
print(std,mean)

PS4 = np.array([df.loc[df['Platform'] == 'PS4', 'Global_Sales']])
std = PS4.std()
mean = PS4.mean()
print(std,mean)

In [ ]:
print(df.isnull().sum())
df_aux = deepcopy(df)

<h4 align = "center"> 1.5 transforming Year to int</h4>
this step will ensure year is an integer for easier computation

In [ ]:
df["Year"] = pd.to_numeric(df["Year"], downcast='integer')
df.info()

<h4 align = "center"> 1.6 categorical data counts</h4>

the next step is to account for the data that is not numerical, since ML models do not do well with categorical data

In [ ]:
print(df["Genre"].unique())
print("_"*100)
print(df["Platform"].unique())
print("_"*100)
print(df["Publisher"].unique())


since there are 31 platforms, 12 Genres and 576 publishers, I am going to use binary encoding<br>
with one hot encoding, the minumum number would be 30+11+575 wich ends up being 616 additional columns, wich is not good.

In [ ]:
print(df["Genre"].value_counts())
print(f"there are {len(pd.unique(df['Genre']))} genres")

In [ ]:
print(df["Platform"].value_counts())
print(f"there are {len(pd.unique(df['Platform']))} platforms")

In [ ]:
print(df["Publisher"].value_counts())
print(f"there are {len(pd.unique(df['Publisher']))} publishers")

<h1 align = "center">2 DATA VIZUALIZATION </H1>

<h4 align = "center"> 2.1 pie chart of game publishers </h4>

In [ ]:
publisher_counts = df['Publisher'].value_counts()
total = publisher_counts.sum()

mask = publisher_counts / total >= 0.02
above = publisher_counts[mask]
below_sum = publisher_counts[~mask].sum()

publisher_counts = pd.concat([above, pd.Series({'Under 2%': below_sum})])

ax = publisher_counts.plot.pie(labels=publisher_counts.index)

<h4 align = "center"> 2.2 pie chart of platforms </h4>

In [ ]:
publisher_counts = df['Platform'].value_counts()
total = publisher_counts.sum()

mask = publisher_counts / total >= 0.02
above = publisher_counts[mask]
below_sum = publisher_counts[~mask].sum()

publisher_counts = pd.concat([above, pd.Series({'Under 2%': below_sum})])

ax = publisher_counts.plot.pie(labels=publisher_counts.index)

<h4 align = "center"> 2.3 pie chart of categories </h4>

In [ ]:
df["Genre"].value_counts().plot.pie()

<h4 align = "center"> 2.4 hist of years</h4>

In [ ]:
df["Year"].plot.hist(bins = 20,edgecolor="black")
plt.xlabel("Year")
plt.title("histogram of years")

<h4 align = "center"> 2.5 histograms of charges</h4>

all of these are left skewed and will be log scaled for more accurate models.

In [ ]:
def histcharge(x, upper_limit = 0.8):
    x.plot.hist(bins=15,edgecolor="black",range=[0,upper_limit])
    arr = np.array(x)
    mean = arr.mean()
    std = arr.std()
    print(mean,std)

In [ ]:
histcharge(df['NA_Sales'])

In [ ]:
histcharge(df['EU_Sales'])

In [ ]:
histcharge(df["JP_Sales"])

In [ ]:
histcharge(df["Other_Sales"])

In [ ]:
histcharge(df["Global_Sales"],2)

In [ ]:
def scatter(sale) : sns.scatterplot(data = df,x = "Year",y=sale)
scatter("NA_Sales")

In [ ]:
scatter("EU_Sales")

In [ ]:
scatter("JP_Sales")

In [ ]:
scatter("Other_Sales")

In [ ]:
scatter("Global_Sales")

<h4 align = "center"> 1.6 Pearson s correlation </h4>

running a pearson s correlation for numeric data alone is a good way to see if I have any global liniarity,despite having some cases of multicolinearity with values of .9 and up (one even hitting 0.94) caused by the fact that global sales is a sum of the other specific sales. <br>
we can do almost nothing with correlations between various sales
since this is missing all the categorical variables, years do not have a high correlation with any other numerical field, and as stated above we cannot do anything with the corr of different sales we will steer away from any sort of linear model

In [ ]:
df.corr(numeric_only=True)

<h4 align = "center"> 1.8 label encoding </h4><br>
despite the fact that this is not the best and proper way to encode nominal categorical data that, for the sake of a peek at what is and is not relevant it is sufficient.

label encoding for the genie impurity, for information gain, and mutual information

In [ ]:
df_analysis = df[['Genre', 'Platform', 'Publisher','Year', 'Name','Global_Sales']].copy()

le = LabelEncoder()
for col in ['Genre', 'Platform', 'Publisher','Name']: df_analysis[col] = le.fit_transform(df_analysis[col].astype(str))
df_analysis.head()

<h4 align = "center"> 1.9 information gain </h4>

we will first use information gain in order to know if there is a relationship between eace feauture and the target, it is based on "entropy estimation from k-nearest neighbors distances" (sklearn mutual_info_regression documentation), this will help decide the feautures used in the actual model training and testing

In [ ]:
df_analysis = df_analysis.dropna()
y = df_analysis['Global_Sales']
x = df_analysis.drop(['Global_Sales'], axis = 1)

mi_scores = mutual_info_regression(x, y, random_state=42)

mi_results = pd.Series(mi_scores, index=x.columns).sort_values(ascending=True)
mi_results.plot.bar()
print(mi_results)

after doing a mutual information, we find out publisher is the only one with a moderate relationship. this show us that the publisher is at least twice more informative than the platform. This could show us that brand loyalty and good previous results will improve the probabily of earning bigger sums than the genre or the platform the game is on

<h4 align = "center"> 1.10 feauture importance
 </h4>

this is a mean increase in impurity feauture importance calculation, it is defined as "the mean and standard deviation of accumulation of the impurity decrease within each tree." ( sklearn Feature importances with a forest of trees), this basically shows how much each feautures reduces entropy.<br>
as stated on sklearn " Impurity-based feature importances can be misleading for high cardinality ", since in this df publisher is the one with the highest cardinality, and it has less than 5%, we will not go the extra step for feauture permutation


In [ ]:
from sklearn.tree import DecisionTreeRegressor



model = DecisionTreeRegressor(max_depth=5, random_state=42)
model.fit(x, y)

combined_importance = pd.Series(model.feature_importances_, index=x.columns).sort_values(ascending=True)
combined_importance.plot.bar()
print(combined_importance)

we find that the name and genre are way more relevant when testing feauture importance than the publisher meaning the model found a relationship between genre and name

<h1 align = "center"> 2 Tuned models

In [ ]:
df.head()
df = df.dropna()

In [ ]:
print(df.isnull().sum())

In [42]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn import metrics
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from category_encoders import BinaryEncoder
from sklearn.multioutput import MultiOutputRegressor


base_models = {
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Extra Trees": ExtraTreesRegressor(),
    "XGBoost": MultiOutputRegressor(XGBRegressor())
}

param_grids = {
    "Decision Tree": {
        'max_depth': [3, 5, 8, 12, 15, None],
        'min_samples_split': randint(2, 21),
        'min_samples_leaf': randint(1,5),
        'criterion': ['squared_error', 'friedman_mse'],
        'ccp_alpha': uniform(0.0, 0.05)
    },
    "Random Forest": {
        'n_estimators': randint(100,801),
        'max_depth': randint(1,15),
        'min_samples_leaf': randint(1,5),
        'max_features': ['sqrt', 'log2'],
        'bootstrap': [True]
    },
    "Extra Trees": {
        'n_estimators': randint(100,501),
        'max_depth': [5, 10, 20, None],
        'min_samples_split': randint(2, 11),
        'bootstrap': [True, False]
    },
    "XGBoost": {
    'estimator__n_estimators':     randint(100, 1001),
    'estimator__learning_rate':    uniform(0.01, 0.09),
    'estimator__max_depth':        randint(3, 10),
    'estimator__subsample':        uniform(0.6, 0.4),
    'estimator__colsample_bytree': uniform(0.6, 0.4),
    'estimator__gamma':            uniform(0, 0.2)
    }
}


encoder = BinaryEncoder(cols=["Publisher", "Name", "Platform", "Genre"])
df = encoder.fit_transform(df)

x = df.drop(columns = ["NA_Sales","EU_Sales","JP_Sales","Other_Sales","Global_Sales"])
y = df[["NA_Sales","EU_Sales","JP_Sales","Other_Sales","Global_Sales"]]
y = np.log1p(y)

x_train33,x_test33,y_train33,y_test33 = train_test_split(x,y,test_size=0.3,random_state=42)


print(f"{'Model':<25} | {'Train R2':<10} | {'Test R2':<10} | {'Variance':<10} |  {'bias':<10}| {'MAE':<10}")
print("-" * 100)

for name,base in base_models.items():
    params = param_grids[name]

    randomsearch = RandomizedSearchCV(base,params,n_iter=100,cv=5,scoring = 'r2',n_jobs=-1,random_state=42)
    model = randomsearch.fit(x_train33,y_train33)

    predictions = np.expm1(model.predict(x_test33))


    train_r2 = model.score(x_train33, y_train33)
    r2 = model.score(x_test33, y_test33)

    y_test_actual = np.expm1(y_test33)


    bias = 1 - train_r2
    variance =  train_r2 - r2
    mae = metrics.mean_absolute_error(y_test_actual, predictions)


    print(f"{name:<20}   | {train_r2:.4f}   |   {r2:.4f}   |  {variance:.4f}    |   {bias:.2f}   |   ${mae:.2f}" )



Model                     | Train R2   | Test R2    | Variance   |  bias      | MAE       
----------------------------------------------------------------------------------------------------
Decision Tree          | 0.7074   |   0.6910   |  0.0165    |   0.29   |   $0.09
Random Forest          | 0.9391   |   0.8097   |  0.1294    |   0.06   |   $0.07
Extra Trees            | 0.9819   |   0.8291   |  0.1528    |   0.02   |   $0.05
XGBoost                | 0.9458   |   0.8658   |  0.0800    |   0.05   |   $0.05


<h4 align = "center"> 2 preparing the data pipeline </h4>

<h4 align = "center"> 2 separating target and feautures + numerical and categorical data </h4>
notice that rank might tell the model information about the targets, for this reason we will be removing the rank

In [43]:
df_aux = deepcopy(df)
df_aux["Year"] = pd.to_numeric(df_aux["Year"], downcast='integer') # stays float because of nulls
df_aux = df_aux.drop(["Rank"], axis =  1)
df_aux.info()

<class 'pandas.DataFrame'>
Index: 16341 entries, 0 to 16597
Data columns (total 38 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Name_0        16341 non-null  int64  
 1   Name_1        16341 non-null  int64  
 2   Name_2        16341 non-null  int64  
 3   Name_3        16341 non-null  int64  
 4   Name_4        16341 non-null  int64  
 5   Name_5        16341 non-null  int64  
 6   Name_6        16341 non-null  int64  
 7   Name_7        16341 non-null  int64  
 8   Name_8        16341 non-null  int64  
 9   Name_9        16341 non-null  int64  
 10  Name_10       16341 non-null  int64  
 11  Name_11       16341 non-null  int64  
 12  Name_12       16341 non-null  int64  
 13  Name_13       16341 non-null  int64  
 14  Platform_0    16341 non-null  int64  
 15  Platform_1    16341 non-null  int64  
 16  Platform_2    16341 non-null  int64  
 17  Platform_3    16341 non-null  int64  
 18  Platform_4    16341 non-null  int64  
 19 

we will be splitting the df into targets Y and feautures X <br>
we will also split numerical and categorical data in order to prepare the feautures independently

In [44]:
X = df_aux.drop(columns = ["NA_Sales","EU_Sales","JP_Sales","Other_Sales","Global_Sales"])
Y = df_aux[["NA_Sales","EU_Sales","JP_Sales","Other_Sales","Global_Sales"]]

numerical = ["Year"]
categorical = ['Name','Platform','Publisher','Genre']


In [45]:
X.info()
Y.info()

<class 'pandas.DataFrame'>
Index: 16341 entries, 0 to 16597
Data columns (total 33 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Name_0       16341 non-null  int64
 1   Name_1       16341 non-null  int64
 2   Name_2       16341 non-null  int64
 3   Name_3       16341 non-null  int64
 4   Name_4       16341 non-null  int64
 5   Name_5       16341 non-null  int64
 6   Name_6       16341 non-null  int64
 7   Name_7       16341 non-null  int64
 8   Name_8       16341 non-null  int64
 9   Name_9       16341 non-null  int64
 10  Name_10      16341 non-null  int64
 11  Name_11      16341 non-null  int64
 12  Name_12      16341 non-null  int64
 13  Name_13      16341 non-null  int64
 14  Platform_0   16341 non-null  int64
 15  Platform_1   16341 non-null  int64
 16  Platform_2   16341 non-null  int64
 17  Platform_3   16341 non-null  int64
 18  Platform_4   16341 non-null  int64
 19  Year         16341 non-null  int16
 20  Genre_0      16341 non

for numerical values, in this case year alone, simple imputer will be used, it will "Replace missing values using a descriptive statistic","If “most_frequent”, then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value" (sklearn SimpleImputer) <br>
the reason I am using 2 pipelines is due to the fact that I want to account for year too.

In [46]:
from category_encoders import BinaryEncoder
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="constant", fill_value="none")),
    ('encoder', BinaryEncoder(), categorical)
])

In [47]:
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical),
    ("cat", categorical_pipeline, categorical),
])

In [48]:
from sklearn.multioutput import MultiOutputRegressor
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MultiOutputRegressor(RandomForestRegressor(random_state=42)))
])

In [49]:
x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.3)
print(x_train.shape)
print(y_train.shape)

(11438, 33)
(11438, 5)


In [50]:
pipeline.fit(x_train, y_train)
#predict = pipeline.predict(x_test)
accuracy = pipeline.score(x_test,y_test)
print(accuracy)

ValueError: A given column is not a column of the dataframe

In [ ]:
#scatterplot of sales per year, we are trying to see any linear trends in our data in order to decide if we want to use a linear model or not
plt.figure()
sns.scatterplot(data = df, x="Year", y="Global_Sales")
plt.show()


In [ ]:
#using regplot you can get a general ideea of how a line will fit to you data, in this case the line is almost straight and we can observe heteroscedasticity, the spread is not consistent, from 2005- 2016 the has been a rise in sales, but the line is not tilted upwords in our case since there are many datapoints close to 0

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df['Global_Sales'] = pd.to_numeric(df['Global_Sales'], errors='coerce')


plt.figure(figsize=(10, 6))
sns.regplot(data=df, x="Year", y="Global_Sales",line_kws={"color": "red"})
plt.show()

In [ ]:
x = np.array(df["Platform"].value_counts())
x.sort()
print(x)
counts = df["Platform"].value_counts()
mapping = df["Platform"].map(counts) <=10
final = mapping.values.tolist()
y =  df.iloc[final]
print(len(y))